# LLM Evaluation — Without Description

Predict loan outcomes using an LLM with **structured features only** (no borrower description).
Compare results against the XGBoost model on the same 100 test samples.

## Setup

In [1]:
import sys
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

from llm_utils import (
    load_llm_sample, run_ml_on_sample,
    build_system_prompt, build_user_prompt,
    call_llm, parse_llm_response,
    evaluate_predictions, compare_results,
    RESULTS_DIR
)

In [2]:
# ── Configuration ──────────────────────────────────────────────────────────
# API key loaded automatically from .env file
API_PROVIDER = "gemini"            # "gemini", "anthropic", or "openai"
MODEL_NAME   = "gemini-2.5-flash"  # None = use default for provider
API_KEY      = None                # None = read from .env / environment

## Load Data & Run XGBoost

In [3]:
llm_sample = load_llm_sample()
y_true = llm_sample['loan_status'].values

print(f"Sample size: {len(llm_sample)}")
print(f"Class distribution:\n{llm_sample['loan_status'].value_counts()}")

Sample size: 100
Class distribution:
loan_status
1    84
0    16
Name: count, dtype: int64


In [4]:
xgb_probs, xgb_preds = run_ml_on_sample(llm_sample)
print(f"XGBoost predictions ready: {len(xgb_preds)} samples")

XGBoost predictions ready: 100 samples


## LLM Predictions (No Description)

In [5]:
system_prompt = build_system_prompt()

# Preview the prompt for the first loan
sample_prompt = build_user_prompt(llm_sample.iloc[0], include_desc=False)
print("System prompt:")
print(system_prompt)
print("\n" + "=" * 50)
print("\nSample user prompt:")
print(sample_prompt)

System prompt:
You are a credit risk analyst. Given a loan application's features, predict whether the borrower will fully repay the loan or default (charge off).

Respond ONLY with valid JSON in this exact format:
{"prediction": <1 or 0>, "reasoning": "<brief explanation>"}

Where:
- prediction: 1 = Fully Paid, 0 = Charged Off
- reasoning: 1-2 sentence explanation of your prediction


Sample user prompt:
Predict the outcome for this loan application:

- Loan amount requested ($): 10000.0
- Loan term:  36 months
- Interest rate (%): 12.12
- Monthly payment ($): 332.72
- LC assigned loan grade: B
- LC assigned loan sub-grade: B3
- Home ownership status: RENT
- Annual income ($): 82000.0
- Income verification status: Not Verified
- Stated loan purpose: credit_card
- Debt-to-income ratio: 3.57
- Earliest credit line date: 1983-09-01
- Number of open credit accounts: 6.0
- Has derogatory public records (0/1): 0
- Revolving balance ($): 6990.0
- Revolving utilization rate (%): 61.3
- Total 

In [6]:
# Run LLM on all 100 samples (resumable — skips already-completed rows)
if 'llm_predictions' not in dir() or not llm_predictions:
    llm_predictions = []
    llm_reasonings = []
    llm_raw_responses = []

start_from = len(llm_predictions)
if start_from > 0:
    print(f"Resuming from sample {start_from}/{len(llm_sample)}")

for i, (_, row) in enumerate(tqdm(llm_sample.iterrows(), total=len(llm_sample))):
    if i < start_from:
        continue

    user_prompt = build_user_prompt(row, include_desc=False)

    raw = call_llm(
        system_prompt, user_prompt,
        api_provider=API_PROVIDER, model=MODEL_NAME, api_key=API_KEY
    )
    llm_raw_responses.append(raw)

    parsed = parse_llm_response(raw)
    llm_predictions.append(parsed['prediction'])
    llm_reasonings.append(parsed['reasoning'])

print(f"\nCompleted: {len(llm_predictions)} predictions")
print(f"Parse errors: {sum(1 for p in llm_predictions if p is None)}")

  6%|▌         | 6/100 [01:01<13:42,  8.75s/it]

  Retry 1/5 in 5s...


100%|██████████| 100/100 [12:12<00:00,  7.33s/it]


Completed: 100 predictions
Parse errors: 0


## Evaluation

In [7]:
llm_metrics = evaluate_predictions(y_true, llm_predictions, label="LLM (No Desc)")
xgb_metrics = evaluate_predictions(y_true, xgb_preds.tolist(), label="XGBoost")


LLM (No Desc) Results (100 samples)
Accuracy: 56.0%

Classification Report:
              precision    recall  f1-score   support

 Charged Off       0.21      0.62      0.31        16
  Fully Paid       0.88      0.55      0.68        84

    accuracy                           0.56       100
   macro avg       0.55      0.59      0.49       100
weighted avg       0.78      0.56      0.62       100

Confusion Matrix:
[[10  6]
 [38 46]]

XGBoost Results (100 samples)
Accuracy: 71.0%

Classification Report:
              precision    recall  f1-score   support

 Charged Off       0.29      0.56      0.38        16
  Fully Paid       0.90      0.74      0.81        84

    accuracy                           0.71       100
   macro avg       0.59      0.65      0.60       100
weighted avg       0.80      0.71      0.74       100

Confusion Matrix:
[[ 9  7]
 [22 62]]


In [8]:
comparison = compare_results(y_true, llm_predictions, xgb_preds.tolist(), llm_reasonings)

print(f"LLM accuracy: {comparison['llm_correct'].mean()*100:.1f}%")
print(f"XGBoost accuracy: {comparison['xgb_correct'].mean()*100:.1f}%")
print(f"\nAgreement between LLM and XGBoost: {(comparison['llm_pred'] == comparison['xgb_pred']).mean()*100:.1f}%")

comparison.head(10)

LLM accuracy: 56.0%
XGBoost accuracy: 71.0%

Agreement between LLM and XGBoost: 79.0%


,actual,llm_pred,xgb_pred,llm_correct,xgb_correct,llm_reasoning
0,1,1,1,1,1,The borrower has an exceptionally low debt-to-...
1,1,0,0,0,0,The borrower exhibits significant financial di...
2,1,0,0,0,0,Despite a very low debt-to-income ratio and ex...
3,1,0,0,0,0,"The borrower has a low annual income ($30,000)..."
4,1,0,1,0,1,"The applicant presents significant red flags, ..."
5,1,0,0,0,0,Despite a good debt-to-income ratio and long c...
6,1,1,1,1,1,"The borrower has a strong credit profile, char..."
7,1,1,1,1,1,The borrower exhibits strong creditworthiness ...
8,1,0,1,0,1,The extremely high revolving utilization rate ...
9,1,1,1,1,1,"The borrower has a very long credit history, a..."


In [9]:
# Cases where LLM and XGBoost disagree
disagree = comparison[comparison['llm_pred'] != comparison['xgb_pred']]
print(f"Disagreements: {len(disagree)} / {len(comparison)}")
print(f"LLM correct in disagreements: {disagree['llm_correct'].sum()}")
print(f"XGBoost correct in disagreements: {disagree['xgb_correct'].sum()}")

if len(disagree) > 0:
    print("\nSample disagreements with LLM reasoning:")
    for _, row in disagree.head(5).iterrows():
        actual = 'Fully Paid' if row['actual'] == 1 else 'Charged Off'
        llm = 'Fully Paid' if row['llm_pred'] == 1 else 'Charged Off'
        xgb = 'Fully Paid' if row['xgb_pred'] == 1 else 'Charged Off'
        print(f"  Actual: {actual} | LLM: {llm} | XGBoost: {xgb}")
        print(f"  Reasoning: {row['llm_reasoning']}\n")

Disagreements: 21 / 100
LLM correct in disagreements: 3
XGBoost correct in disagreements: 18

Sample disagreements with LLM reasoning:
  Actual: Fully Paid | LLM: Charged Off | XGBoost: Fully Paid
  Reasoning: The applicant presents significant red flags, including an extremely high revolving utilization rate of 79.7% and a large revolving balance relative to their low, unverified annual income. These factors, combined with the loan's purpose for debt consolidation, indicate severe financial strain and a high risk of default.

  Actual: Fully Paid | LLM: Charged Off | XGBoost: Fully Paid
  Reasoning: The extremely high revolving utilization rate of 81.4% and a low annual income of $29,000 strongly suggest financial strain and difficulty managing existing debt. Despite a long credit history, these factors, coupled with the LC grade of C3 and unverified income, indicate a high likelihood of default.

  Actual: Fully Paid | LLM: Charged Off | XGBoost: Fully Paid
  Reasoning: Despite a goo

## Export Results

In [10]:
os.makedirs(RESULTS_DIR, exist_ok=True)

comparison.to_csv(f"{RESULTS_DIR}/04a_llm_no_desc_results.csv", index=False)

summary = pd.DataFrame([llm_metrics, xgb_metrics],
                        index=['LLM (No Desc)', 'XGBoost'])
summary.to_csv(f"{RESULTS_DIR}/04a_llm_no_desc_metrics.csv")
print(summary.to_string())

               accuracy  precision_charged_off  recall_charged_off  f1_charged_off  n_valid
LLM (No Desc)      0.56               0.208333              0.6250        0.312500      100
XGBoost            0.71               0.290323              0.5625        0.382979      100
